# Intro til Machine Learning — 3: MNIST og Pokémon — hele netværk

Finalen! I skal træne et neuralt netværk til at genkende **håndskrevne tal** — rigtige
billeder, skrevet af rigtige (amerikanske) mennesker. Datasættet hedder **MNIST** og er
ML-verdenens svar på "Hello World": 70.000 små gråtonebilleder af cifrene 0–9.

Og her er dagens vigtigste pointe, før vi overhovedet starter: **et billede kan skrives
som tal i en tabel.** Derfor virker hele jeres værktøjskasse — pandas, standardisering,
tensorer, klasser, træningsloops, softmax — direkte på billeder. Lad os læse nogle tal!

Og bagefter vender vi tilbage til **Pokémon** (afsnit 3) og træner et helt netværk til at
gætte typer ud fra stats — samme maskineri, ny slags data, og en lærerig lektion i, hvornår
en model *ikke* kan finde svaret.

> **Om opgaverne:** Der er med vilje flere opgaver, end du kan nå — du behøver ikke nå alt. Opgaver mærket **(find fejlen)** har en bevidst fejl, som du skal finde og rette (så en fejl dér er meningen). Nederst i notebooken ligger et par **ekstra opgaver**, hvis du får lyst til mere.
>
> Noget af det her er nyt og kan føles udfordrende i starten — og det er helt okay. Vi forklarer hvert skridt så klart og tydeligt, vi kan, og der er et hint til hver opgave, hvis du går i stå. Tag dig endelig god tid.

## Setup

In [ ]:
# Henter MNIST (nedskaleret) + Pokémon fra GitHub (Plan B: upload filerne via mappeikonet)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/mnist_traen_lille.csv.gz
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/mnist_test_lille.csv.gz
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/Pokemon.csv

!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/98-Helpers/helpers.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from helpers import show_mnist_images

torch.manual_seed(42)
np.random.seed(42)

# 1: Billeder er tal (i en DataFrame)

Vi henter MNIST fra GitHub — **som CSV-fil**, så I kan se med egne øjne, at et billede
også er en tabel af tal. (Vi bruger en nedskaleret udgave med 16.000 billeder, så det kører hurtigt.)

In [ ]:
train_df = pd.read_csv("mnist_traen_lille.csv.gz")
test_df = pd.read_csv("mnist_test_lille.csv.gz")
print("træning:", train_df.shape)
print("test:   ", test_df.shape)

> **Plan B:** Henter `wget`-cellen ikke filerne, så upload `mnist_traen_lille.csv.gz` og
> `mnist_test_lille.csv.gz` manuelt via mappeikonet i Colab (de ligger i `28-Data/MLData` på GitHub).

**(16000, 785)** — en tabel med 16.000 rækker, én pr. billede. Og de 785 kolonner?
Den første er `label` (hvilket ciffer billedet forestiller), og resten er... pixels!
$28 \times 28 = 784$ pixels pr. billede, foldet ud i én lang række, med gråtoneværdier
fra 0 (sort) til 255 (hvid). Det er altså en stor tabel, ligesom Pokémon-tabellen — nu
med flere kolonner. Vi bruger 16.000 trænings- og 2.000 testbilleder — rigeligt til CPU'en.

Nu deler vi tabellen i **y** (første kolonne: cifret) og **X** (de 784 pixels) — og
ser det første billede som det, det i virkeligheden er: en bunke tal.

In [ ]:
y_train_np = train_df["label"].values
X_train_np = train_df.drop(columns=["label"]).values     # alt undtagen label-kolonnen

row = X_train_np[0]
print("én række:", row.shape, "— min:", row.min(), " max:", row.max())
print("label:", y_train_np[0])

In [ ]:
image = row.reshape(28, 28)      # 784 tal → 28×28-gitter (reshape fra opgave 5.5!)

plt.imshow(image, cmap="gray")
plt.title(f"label: {y_train_np[0]}")
plt.colorbar(label="pixelværdi")
plt.show()

Der er cifret! `show_mnist_images` fra hjælpefilen viser et helt grid ad gangen:

In [ ]:
show_mnist_images(X_train_np[:10], y_train_np[:10], n=10)

## Klargøring: normalisér og konvertér

Pixelværdier på 0–255 er alt for store tal til et netværk (regressionsforløbet-lektien!). Her er
alle features heldigvis på SAMME skala, så vi kan nøjes med at dele med 255 — så ligger
alt mellem 0 og 1. Derefter laver vi tensorer. Bemærk `torch.long` til labels —
klassenumre skal være hele tal (CrossEntropy-reglen fra opgave 12.6!):

In [ ]:
X_train = torch.tensor(X_train_np / 255.0, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.long)

X_test = torch.tensor(test_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
y_test = torch.tensor(test_df["label"].values, dtype=torch.long)

print(X_train.shape, X_train.min().item(), "til", X_train.max().item())

### Opgaver

##### Opgave 1.1
Vi starter blidt: et MNIST-billede er en række tal, og her kigger vi på dem ét ad gangen.

Prøv at køre cellen, og skift så `i` til et andet tal og bladr lidt rundt — der er 16.000 billeder at vælge imellem.

Se om du kan finde et ciffer, der er skrevet så sjusket, at du selv er i tvivl om, hvad det forestiller. Passer labelen med det, du ville have gættet?

Hint: `i` er rækkenummeret i tabellen — prøv fx `i = 42` eller `i = 1234` og kør igen.

In [ ]:
i = 0   # ← bladr med denne
plt.imshow(X_train_np[i].reshape(28, 28), cmap="gray")
plt.title(f"label: {y_train_np[i]}")
plt.show()

##### Opgave 1.2
Et billede ligger i tabellen som én lang række på 784 tal. For at se det som et billede skal vi folde rækken ud i et kvadratisk gitter.

Prøv at udfylde det sidste tal i `reshape`, så de 784 tal bliver til et 28×28-billede.

Husk: $28 \times 28 = 784$ — begge tal skal gå op i det samlede antal pixels.

In [ ]:
row = X_train_np[7]
image = row.reshape(28, ...)      # ← udfyld det sidste tal: 28 rækker × 28 kolonner
plt.imshow(image, cmap="gray")
plt.show()
# skal vise ét håndskrevet ciffer

##### Opgave 1.3
Vi ser på, om cifrene 0–9 optræder lige tit i træningssættet — altså om klasserne er lige store.

Cellen bruger `value_counts()` på label-kolonnen `train_df["label"]`. Prøv at køre den og kig på tallene: er der nogenlunde lige mange af hvert ciffer, eller stikker nogle ud?

Tænk også over, hvorfor det er værd at tjekke — en model kan opnå høj accuracy ved altid at gætte det hyppigste ciffer, hvis klasserne er meget skæve.

Hint: hvis ét ciffer fyldte 90 % af dataene, hvad ville den dovne model så gætte hver gang?

In [ ]:
train_df["label"].value_counts().sort_index()

##### Opgave 1.4
Nedenfor normaliserer vi pixelværdierne. De ligger fra 0 til 255, og så store tal er svære for et netværk at arbejde med.

Prøv at udfylde tallet, vi deler med, så alle pixels havner mellem 0 og 1.

Tænk bagefter over: hvorfor kan vi her nøjes med at dele med det største tal i stedet for den fulde standardisering fra notebook 1? Kig på, at alle pixels allerede er på præcis samme skala, 0–255.

Hint: hvad er den lyseste pixelværdi et billede kan have — og hvad giver 255/255?

In [ ]:
X_alternativ = torch.tensor(X_train_np / ..., dtype=torch.float32)   # ← udfyld: del med den største pixelværdi
print(X_alternativ.min().item(), "til", X_alternativ.max().item())
# skal printe noget i stil med "0.0 til 1.0"

##### Opgave 1.5 (find fejlen)
Her prøver vi at vise et billede direkte fra tabellen, men `reshape` crasher med *"cannot reshape array of size 785 into shape (28,28)"*. Et billede har jo 784 pixels — hvor kommer det 785. tal fra?

Prøv at læse koden og ret den, så du kun tager pixel-kolonnerne med (ikke hele rækken).

Husk: den allerførste kolonne i tabellen er `label`, ikke en pixel — det er derfor rækken har 785 tal og ikke 784. Kig på, hvordan vi fjernede den med `drop(columns=["label"])` i Setup.

In [ ]:
row = train_df.values[0]          # første række fra tabellen
plt.imshow(row.reshape(28, 28), cmap="gray")
plt.show()

##### Opgave 1.7
Vi tæller, hvor mange pixels ét MNIST-billede består af. Billedet er gemt som én lang række tal.

Prøv at udfylde linjen, så den printer antallet af pixels i rækken.

Hint: `image` er en almindelig talrække — hvad giver `len(image)` (eller `image.shape`)?

In [ ]:
image = X_train_np[0]
print("antal pixels:", ...)          # ← udfyld: brug len(image) eller image.shape
print("lyseste pixel:", image.max())
# antal pixels skal give 784

# 2: Netværket der kan læse

Arkitekturen: **784 pixels ind → 128 skjulte neuroner (ReLU) → 10 point ud** (ét pr.
ciffer). Ingen softmax i modellen — vi bruger `nn.CrossEntropyLoss`, og den vil have de rå
point (samme regel som i aktiveringsfunktions-notebooken: `CrossEntropyLoss` kører selv softmax indeni).

Én ny ting: **mini-batches**. I stedet for at vise netværket alle 16.000 billeder før hvert
skridt, viser vi det 64 ad gangen og tager et skridt pr. portion. Det giver mange flere
(og lidt mere støjede) skridt pr. epoke — i praksis træner det både hurtigere og bedre.
Læg mærke til, at det er endnu en for-løkke med slicing:

In [ ]:
class DigitNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 128)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(128, 10)    # 10 klasser → 10 rå point (ingen softmax her!)

    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))

model = DigitNet()
print(model)
print("parametre:", sum(p.numel() for p in model.parameters()))

In [ ]:
model = DigitNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
batch_size = 64

for epoch in range(15):
    for i in range(0, len(X_train), batch_size):      # 64 billeder ad gangen
        X_batch = X_train[i:i + batch_size]
        y_batch = y_train[i:i + batch_size]

        optimizer.zero_grad()                               # de fem trin:
        loss = loss_fn(model(X_batch), y_batch)         # nulstil → forward+tab
        loss.backward()                                      # → backward
        optimizer.step()                                    # → step

    print(f"epoke {epoch + 1}: tab på sidste portion = {loss.item():.4f}")

## Eksamen

10 point pr. billede — `argmax(dim=1)` finder indekset med flest point, altså modellens
gæt. Resten kender I:

In [ ]:
with torch.no_grad():
    point = model(X_test)

pred = point.argmax(dim=1)
accuracy = (pred == y_test).float().mean()
print(f"test-accuracy: {accuracy.item():.1%}")

Et netværk på ~100.000 parametre, trænet på et par minutter, læser håndskrift med
~95 % sikkerhed. Lad os kigge det i kortene: softmax (NU må vi gerne — vi skal kun *vise*
sandsynligheder, ikke træne) laver de 10 point om til en fordeling:

In [ ]:
i = 0
with torch.no_grad():
    point_i = model(X_test[i:i + 1])
probabilities = torch.softmax(point_i, dim=1).squeeze()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].imshow(X_test[i].reshape(28, 28), cmap="gray")
axes[0].set_title(f"label: {y_test[i].item()}")
axes[0].axis("off")
axes[1].bar(range(10), probabilities)
axes[1].set_xticks(range(10))
axes[1].set_title("modellens sandsynligheder")
axes[1].set_xlabel("ciffer")
plt.show()

### Opgaver

##### Opgave 2.1
Vi undersøger, hvor meget netværkets størrelse betyder. `FlexDigitNet` tager antallet af skjulte neuroner som parameter og tager samtidig tid på træningen.

Prøv at træne den med 32, 128 og 512 skjulte neuroner ved at skrue på `hidden`. Notér accuracy og træningstid for hver.

Tænk over: kan det betale sig at blive ved med at gøre netværket større?

Hint: kig på begge tal — bliver accuracy ved med at stige lige så meget, som træningstiden gør?

In [ ]:
import time

class FlexDigitNet(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.layer1 = nn.Linear(784, hidden)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(hidden, 10)

    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))

hidden = 32   # ← prøv 32, 128 og 512
start = time.time()
model_f = FlexDigitNet(hidden)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_f.parameters(), lr=0.001)
for epoch in range(5):
    for i in range(0, len(X_train), 64):
        optimizer.zero_grad()
        loss = loss_fn(model_f(X_train[i:i + 64]), y_train[i:i + 64])
        loss.backward()
        optimizer.step()

with torch.no_grad():
    acc = (model_f(X_test).argmax(dim=1) == y_test).float().mean()
print(f"{hidden} skjulte: accuracy = {acc.item():.1%}, tid = {time.time() - start:.1f} s")

##### Opgave 2.2
Nu skal du selv skrive træningsloopets rytme. Skabelonen henter en portion billeder ad gangen; du udfylder de fire linjer, der træner på portionen.

Prøv at udfylde de fire trin i den faste rækkefølge: nulstil gradienter → gæt + beregn tab → regn baglæns → tag et skridt.

Husk: det er præcis samme rytme som træningsloopet i cellerne ovenfor — kig efter `zero_grad`, `backward` og `step`.

In [ ]:
model2 = DigitNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model2.parameters(), lr=0.001)

for epoch in range(5):
    for i in range(0, len(X_train), 64):
        X_batch = X_train[i:i + 64]
        y_batch = y_train[i:i + 64]
        ...   # ← 1. nulstil gradienterne
        ...   # ← 2. gæt på X_batch og beregn tabet mod y_batch — gem det i loss
        ...   # ← 3. regn baglæns (find gradienterne)
        ...   # ← 4. tag ét skridt med optimizeren

with torch.no_grad():
    acc = (model2(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy: {acc.item():.1%}")

##### Opgave 2.3
Vi kigger på de billeder, modellen tager fejl af. Skabelonen finder dem, og `show_mnist_images` viser modellens gæt i rødt.

Prøv at køre cellen og studere de ti fejlgæt. Er der nogen, hvor du godt kan forstå modellens gæt — eller hvor gættet nærmest er bedre end labelen?

Hint: mange fejl sker på cifre, der ligner hinanden — kig efter 4/9, 3/5 eller 7/1.

In [ ]:
forkerte = (pred != y_test).nonzero().squeeze()
print("antal fejlgæt:", len(forkerte), "af", len(y_test))

show_mnist_images(X_test[forkerte[:10]], y_test[forkerte[:10]],
                   predictions=pred[forkerte[:10]], n=10)

##### Opgave 2.4
Vi bytter aktiveringsfunktionen ud og ser, om det ændrer noget. Netværket her er lavvandet (kun ét skjult lag).

Prøv at skifte `nn.ReLU()` ud med `nn.Sigmoid()` i `SigmoidDigitNet` og træn igen i 5 epoker. Sammenlign accuracy med ReLU-udgaven.

Tænk over, hvorfor forskellen er lille her, men bliver dramatisk i et dybt netværk med mange lag.

Hint: sigmoid maser store tal sammen tæt på 0 og 1 — i mange lag oven på hinanden kan gradienten næsten forsvinde, men med kun ét skjult lag mærkes det knap.

In [ ]:
class SigmoidDigitNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 128)
        self.activation = nn.ReLU()     # ← skift til nn.Sigmoid()
        self.layer2 = nn.Linear(128, 10)

    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))

model_s = SigmoidDigitNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_s.parameters(), lr=0.001)
for epoch in range(5):
    for i in range(0, len(X_train), 64):
        optimizer.zero_grad()
        loss = loss_fn(model_s(X_train[i:i + 64]), y_train[i:i + 64])
        loss.backward()
        optimizer.step()

with torch.no_grad():
    acc = (model_s(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy: {acc.item():.1%}")

##### Opgave 2.5
Vi kigger på, hvor sikker modellen er, når den tager fejl. `softmax` laver de 10 rå point om til sandsynligheder, som vi kan plotte.

Prøv at udfylde `dim` i softmax-kaldet, så det summer hen over de 10 klasser. Cellen bruger allerede `forkerte[0]` — et af fejlgættene fra opgave 2.3.

Sammenlign søjlerne med et sikkert gæt: er modellen i tvivl mellem to cifre, eller helt skæv?

Hint: outputtet har formen `(1, 10)` — hvilken akse (`dim`) skal softmax summe over, så de 10 tal tilsammen giver 1?

In [ ]:
i = forkerte[0].item()   # et billede modellen tog fejl af
with torch.no_grad():
    point_i = model(X_test[i:i + 1])
probabilities = torch.softmax(point_i, dim=...).squeeze()   # ← udfyld: 1 (summér hen over de 10 klasser)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].imshow(X_test[i].reshape(28, 28), cmap="gray")
axes[0].set_title(f"label: {y_test[i].item()}, gæt: {pred[i].item()}")
axes[0].axis("off")
axes[1].bar(range(10), probabilities)
axes[1].set_xticks(range(10))
plt.show()

##### Opgave 2.6
Nu ser vi overfitting med egne øjne. Skabelonen træner netværket på kun de første 500 billeder, men i hele 40 epoker, og måler accuracy på både træningsbillederne og testsættet.

Prøv at køre cellen og sammenlign de to tal. Sæt selv ord på, hvad der er sket, med dine egne ord.

Hint: hvis modellen rammer næsten 100 % på træningsbillederne, men meget lavere på test, hvad har den så gjort — lært mønstre eller lært de 500 billeder udenad?

In [ ]:
X_lille = X_train[:500]
y_lille = y_train[:500]

model_lille = DigitNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_lille.parameters(), lr=0.001)
for epoch in range(40):
    for i in range(0, len(X_lille), 64):
        optimizer.zero_grad()
        loss = loss_fn(model_lille(X_lille[i:i + 64]), y_lille[i:i + 64])
        loss.backward()
        optimizer.step()

with torch.no_grad():
    train_acc = (model_lille(X_lille).argmax(dim=1) == y_lille).float().mean()
    test_acc = (model_lille(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy på de 500 træningsbilleder: {train_acc.item():.1%}")
print(f"accuracy på testsættet:              {test_acc.item():.1%}")

##### Opgave 2.7 (find fejlen)
Din sidemand har "forbedret" netværket ved at lægge en softmax ind i selve modellen — "så den giver pæne sandsynligheder". Men træningen er blevet mystisk sløv, og accuracy er faldet.

Prøv at finde problemet og fjern det, så netværket igen giver rå point ud.

Hint: `nn.CrossEntropyLoss` laver selv softmax internt og vil have de rå point — lægger man en softmax oven i, bliver den anvendt to gange. Kig på setup-teksten om, hvorfor der ikke skal softmax i modellen.

In [ ]:
class BuddyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 128)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(128, 10)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        return self.softmax(self.layer2(self.activation(self.layer1(x))))

model_k = BuddyNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_k.parameters(), lr=0.001)
for epoch in range(5):
    for i in range(0, len(X_train), 64):
        optimizer.zero_grad()
        loss = loss_fn(model_k(X_train[i:i + 64]), y_train[i:i + 64])
        loss.backward()
        optimizer.step()

with torch.no_grad():
    acc = (model_k(X_test).argmax(dim=1) == y_test).float().mean()
print(f"accuracy: {acc.item():.1%} — hmm, lavere end før?")

##### Opgave 2.8
Vi tænker over en svaghed ved netværket: det ser kun 784 tal i én lang række og aner ikke, at pixel 5 og pixel 33 i virkeligheden sidder lige over hinanden i billedet.

Flytter du et ciffer to pixels til højre, ændrer alle 784 tal sig — men for dine øjne er det jo det samme billede.

Prøv at forestille dig, hvordan man kunne udnytte, at pixels har naboer. Hvad skulle et smartere netværk kigge på i stedet for enkelt-pixels?

Hint: tænk på små områder frem for enkelte prikker — hvad nu hvis netværket kiggede på en lille firkant af pixels ad gangen (fx en kant eller en bue)?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

# 3: Pokémon — træn et helt netværk selv

MNIST var billeder. Nu ser vi på en **tabel** i stedet: Pokémon. Pointen er,
at *maskineriet er præcis det samme* — `nn.Module`, træningsloop, `CrossEntropyLoss`. Vi skifter
kun dataene ud.

Opgaven: **gæt en Pokémons primære type (`Type 1`) ud fra dens seks kampstats.** Det er 18
klasser (mod cifrenes 10) — og, som I skal se, en HÅRD opgave. At ikke alt kan læres lige godt,
er også en pointe.

In [ ]:
from sklearn.model_selection import train_test_split

df = pd.read_csv("Pokemon.csv")   # Pokémon-tabellen (hentet med wget i Setup)

stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
types = sorted(df["Type 1"].unique())               # de 18 typer
type_til_id = {t: i for i, t in enumerate(types)}    # tekst -> tal (ligesom en tokenizer)
print(len(types), "typer:", types)

X_raw = df[stats].values.astype("float32")
X_mean = X_raw.mean(axis=0)
X_std = X_raw.std(axis=0)
X_np = (X_raw - X_mean) / X_std              # standardisér (som i notebook 1)
y_np = df["Type 1"].map(type_til_id).values          # 0..17

X_tr, X_te, y_tr, y_te = train_test_split(X_np, y_np, test_size=0.2, random_state=42)
X_train_p = torch.tensor(X_tr); y_train_p = torch.tensor(y_tr, dtype=torch.long)
X_test_p = torch.tensor(X_te); y_test_p = torch.tensor(y_te, dtype=torch.long)
print("træning:", X_train_p.shape, "| 6 stats ind →", len(types), "typer ud")

## Modellen — 6 tal ind, 18 typer ud

Sammenlign med `CifferNet`: dér var det 784 ind → 10 ud. Her er det 6 ind → 18 ud. Ellers *nul*
forskel. Vi træner med **fuld batch** (kun ~640 Pokémon i træningssættet, så vi behøver ikke
minibatches):

In [ ]:
class PokemonNet(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.layer1 = nn.Linear(6, hidden)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(hidden, 18)       # 18 typer ud

    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))

torch.manual_seed(42)
poke_model = PokemonNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(poke_model.parameters(), lr=0.01)

for epoch in range(300):
    optimizer.zero_grad()
    loss = loss_fn(poke_model(X_train_p), y_train_p)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    train_acc = (poke_model(X_train_p).argmax(dim=1) == y_train_p).float().mean()
    test_acc = (poke_model(X_test_p).argmax(dim=1) == y_test_p).float().mean()
print(f"træning: {train_acc.item():.1%}  |  test: {test_acc.item():.1%}  (tilfældigt = {1/18:.1%})")

Læg mærke til de to tal: modellen rammer ~80 % på **træningsdataene**, men kun ~22 % på **test**.
Det svælg ER **overfitting** — live. Netværket har lært træningssættets Pokémon udenad, men typer
kan ikke aflæses sikkert af stats alene (en Water- og en Grass-Pokémon kan have næsten ens tal).

Og alligevel: 22 % er ~4× bedre end tilfældigt (5,6 %). Modellen *har* fanget et mønster — hurtige,
spinkle Pokémon er tit Electric/Flying; tunge, hårde er tit Rock/Steel. Nogle ting kan læres, andre
ikke. (I *Specialiserede Modeller* møder I værktøjerne mod overfitting — og modeller, der slår
neurale netværk på netop tabeller.)

### Opgaver

##### Opgave 3.1
Vi skruer på netværkets størrelse igen — men denne gang på den svære Pokémon-opgave. Kig især på forskellen mellem trænings- og test-accuracy.

Prøv at træne `PokemonNet` med `hidden = 16`, `128` og `256`.

Bliver test bedre af et større netværk — eller vokser svælget mellem træning og test?

Hint: hvis træning stiger, men test står stille (eller falder), hvad er det så, det større netværk lærer — mønstre eller de enkelte trænings-Pokémon udenad?

In [ ]:
hidden = 16   # ← prøv 16, 128 og 256
torch.manual_seed(42)
model = PokemonNet(hidden)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(300):
    optimizer.zero_grad(); loss_fn(model(X_train_p), y_train_p).backward(); optimizer.step()
with torch.no_grad():
    tr = (model(X_train_p).argmax(dim=1) == y_train_p).float().mean()
    te = (model(X_test_p).argmax(dim=1) == y_test_p).float().mean()
print(f"{hidden} skjulte: træning {tr.item():.1%}, test {te.item():.1%}")

##### Opgave 3.2
Sidste træningsloop du skal udfylde — nu på Pokémon-data. Skabelonen mangler de fire linjer, der udgør rytmen.

Prøv at udfylde dem i rækkefølge: nulstil gradienter → beregn tab → regn baglæns → tag et skridt.

Husk: det er præcis samme fire trin som til cifrene — kun modellen og dataene har skiftet navn.

In [ ]:
torch.manual_seed(42)
model = PokemonNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(300):
    ...          # ← 1. nulstil gradienterne
    loss = ...   # ← 2. beregn tabet på X_train_p / y_train_p
    ...          # ← 3. regn baglæns
    ...          # ← 4. tag et skridt
with torch.no_grad():
    te = (model(X_test_p).argmax(dim=1) == y_test_p).float().mean()
print(f"test: {te.item():.1%}")

##### Opgave 3.3 (find fejlen)
Cellen skulle bygge en type-model, men den crasher med *"Target 15 is out of bounds"*. Fejlen kommer, når en label peger på en klasse, som output-laget slet ikke har plads til.

Prøv at læse fejlen og rette den ene ting, der er galt.

Hint: hvor mange typer er der i alt (kig på `len(types)` i Setup) — og hvor mange point giver output-laget ud?

In [ ]:
class WrongNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(6, 64)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(64, 10)

    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))

torch.manual_seed(42)
model = WrongNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(50):
    optimizer.zero_grad(); loss_fn(model(X_train_p), y_train_p).backward(); optimizer.step()
print("trænet!")

##### Opgave 3.4
Vi giver modellen en ekstra ledetråd: `Total` (summen af alle stats) som en 7. feature.

Prøv at udfylde to ting: tilføj `"Total"` til feature-listen, og ret modellens input-størrelse, så den passer til 7 features.

Tænk over: hjælper det på test-accuracy — og hvorfor / hvorfor ikke?

Hint: `Total` er summen af de seks stats, modellen allerede kender — giver det ny information, eller ved modellen det i forvejen?

In [ ]:
stats7 = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed", ...]   # ← tilføj "Total"
X7 = df[stats7].values.astype("float32")
X7 = (X7 - X7.mean(axis=0)) / X7.std(axis=0)
X7_tr, X7_te, y7_tr, y7_te = train_test_split(X7, y_np, test_size=0.2, random_state=42)
X7_train = torch.tensor(X7_tr); y7_train = torch.tensor(y7_tr, dtype=torch.long)
X7_test = torch.tensor(X7_te); y7_test = torch.tensor(y7_te, dtype=torch.long)

class PokemonNet7(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(..., 64)     # ← hvor mange features nu?
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(64, 18)
    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))

torch.manual_seed(42)
model = PokemonNet7()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(300):
    optimizer.zero_grad(); loss_fn(model(X7_train), y7_train).backward(); optimizer.step()
with torch.no_grad():
    te = (model(X7_test).argmax(dim=1) == y7_test).float().mean()
print(f"med Total: test {te.item():.1%}")

##### Opgave 3.5
Vi slår en bestemt Pokémon op og ser modellens top-3 gæt på dens type.

Prøv at udfylde softmax-funktionen, der laver de 18 rå point om til sandsynligheder.

Se bagefter, om modellen er sikker eller splittet — og prøv et par forskellige Pokémon med `name`.

Hint: det er samme funktion, du brugte til cifrene i opgave 2.5 — den der starter med `torch.soft...`.

In [ ]:
name = "Pikachu"     # ← prøv fx "Gyarados", "Snorlax" eller "Mewtwo"
row = df[df["Name"] == name].head(1)
x = torch.tensor((row[stats].values.astype("float32") - X_mean) / X_std)

with torch.no_grad():
    probabilities = ...(poke_model(x), dim=1)[0]    # ← udfyld softmax-funktionen (som i opgave 2.5)
top3 = probabilities.argsort(descending=True)[:3]

print(f"{name} er i virkeligheden: {row['Type 1'].values[0]}")
print("modellens top-3:")
for i in top3:
    print(f"  {types[i]:10} {probabilities[i].item():.1%}")

## Ekstra opgaver

Her er nogle ekstra udfordringer, hvis du er nået hele vejen igennem og har lyst til mere. De bygger videre på det, du allerede har lavet, og du kan tage dem i den rækkefølge, du vil.

##### Ekstra 1
Cellen viser gennemsnitsbilledet af alle 3-taller — altså hvordan et "gennemsnitligt 3-tal" ser ud (lidt spøgelsesagtigt!).

Se om du kan udvide det til alle 10 cifre: brug en for-løkke over `range(10)` og `plt.subplots` til at vise de ti gennemsnitsbilleder i ét grid.

Hint: du har allerede opskriften for ét ciffer i cellen — læg linjerne ind i en løkke, og udskift `3` med løkkevariablen.

In [ ]:
# Skabelon: udfyld filteret, så løkken laver ét gennemsnitsbillede pr. ciffer
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for digit in range(10):
    billeder = X_train_np[y_train_np == ...]    # <-- udfyld: vælg alle billeder med dette ciffer (brug digit)
    ax = axes[digit // 5, digit % 5]
    ax.imshow(billeder.mean(axis=0).reshape(28, 28), cmap="gray")
    ax.set_title(str(digit))
    ax.axis("off")
plt.tight_layout()
plt.show()

##### Ekstra 2
Vi ser på, hvilket ciffer der "fylder mest" på papiret — altså har flest lyse pixels i gennemsnit.

Prøv at køre cellen og aflæs tallene: hvilket ciffer vinder, og hvilket er tyndest? Overvej, om resultatet passer med, hvordan cifrene ser ud.

Hint: et bredt ciffer som 0 eller 8 tegner mange pixels — et smalt som 1 tegner få. Passer tallene med det?

In [ ]:
for digit in range(10):
    images = X_train_np[y_train_np == digit]
    lyse = (images > 50).sum() / len(images)      # lyse pixels i snit
    print(f"ciffer {digit}: {lyse:.0f} lyse pixels i snit")

##### Ekstra 3
Type var svært — prøv en lettere opgave: er en Pokémon legendarisk? Det er kun to klasser (legendarisk / ikke), så modellen skal vælge mellem to muligheder.

Prøv at udfylde antallet af outputklasser i `LegendaryNet`.

Tænk bagefter over: hvorfor bliver accuracy pludselig så høj — og hvad er fælden?

Hint: kig på den sidste linje, der printer andelen af legendariske. Hvis kun ± 8 % er legendariske, hvor højt rammer en model, der altid gætter "ikke legendarisk"?

In [ ]:
y_leg = df["Legendary"].astype(int).values
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(X_np, y_leg, test_size=0.2, random_state=42)
Xl_train = torch.tensor(Xl_tr); yl_train = torch.tensor(yl_tr, dtype=torch.long)
Xl_test = torch.tensor(Xl_te); yl_test = torch.tensor(yl_te, dtype=torch.long)

class LegendaryNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(6, 32)
        self.activation = nn.ReLU()
        self.layer2 = nn.Linear(32, ...)     # ← hvor mange klasser? (legendarisk / ikke)
    def forward(self, x):
        return self.layer2(self.activation(self.layer1(x)))

torch.manual_seed(42)
model = LegendaryNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(300):
    optimizer.zero_grad(); loss_fn(model(Xl_train), yl_train).backward(); optimizer.step()
with torch.no_grad():
    te = (model(Xl_test).argmax(dim=1) == yl_test).float().mean()
print(f"legendarisk-model: test {te.item():.1%}")
print(f"andel legendariske i ALLE data: {y_leg.mean():.1%}")

##### Ekstra 4
Vi finder ud af, hvilke typer modellen oftest forveksler. Skabelonen løber alle fejlgæt igennem og tæller parret (rigtig type, gættet type) op i en dict.

Prøv at udfylde optællings-linjen, så den lægger én til for hvert par.

Hint: brug samme dict-mønster som før — `confusions.get(pair, 0)` henter den nuværende tælling (0 hvis paret er nyt), og så lægger du 1 til.

In [ ]:
with torch.no_grad():
    pred = poke_model(X_test_p).argmax(dim=1)

confusions = {}
for rigtig, g in zip(y_test_p.tolist(), pred.tolist()):
    if rigtig != g:
        pair = (types[rigtig], types[g])
        confusions[pair] = ...          # ← læg 1 til tællingen for dette par

top = sorted(confusions.items(), key=lambda kv: kv[1], reverse=True)[:5]
for (rigtig, guess), antal in top:
    print(f"{rigtig:10} gættet som {guess:10}: {antal} gange")